# Phase 1 Task B — Machine A: EfficientNet-B3 (standalone)

**Model:** EfficientNet-B3 × 3 seeds [42, 123, 456]
**batch_size:** 32 (chạy đơn, không chia GPU)

Chạy notebook này SAU KHI ConvNeXt + MobileNet đã hoàn tất.
Sau khi Machine B xong Swin + ResNet, Machine A rảnh → chạy EfficientNet.

In [ ]:
from pathlib import Path
NOTEBOOK = Path("__file__").resolve()
ROOT = NOTEBOOK
for _ in range(10):
    ROOT = ROOT.parent
    if (ROOT / "pyproject.toml").exists() or (ROOT / ".git").exists():
        break
import sys, os
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print(f"[ROOT] {ROOT}")

In [ ]:
import subprocess, sys, os, getpass
PKGS = ["timm", "albumentations", "scikit-learn", "seaborn", "wandb", "ptflops"]
for pkg in PKGS:
    try: __import__(pkg.replace("-", "_"))
    except: subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
if "WANDB_API_KEY" not in os.environ:
    os.environ["WANDB_API_KEY"] = getpass.getpass("wandb API key: ")
print("[OK] Dependencies ready")

In [ ]:
HF_REPO = "ny0g507h3p/wood-task-B"
DATA_ROOT = ROOT / "data" / "task-B"
from src.data import get_dataloaders
tl, vl, testl = get_dataloaders(str(DATA_ROOT), batch_size=32, num_workers=4, task="taskB")
print(f"[Data] Train={len(tl.dataset)}, Val={len(vl.dataset)}, Test={len(testl.dataset)}")
print(f"[Data] Classes={len(tl.dataset.classes)}")

In [ ]:
import wandb, torch, numpy as np
from src.models import create_model
from src.train import TrainingEngine, create_optimizer, create_scheduler, create_loss_function
from src.utils import set_seed

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[Device] {DEVICE}")

In [ ]:
def train_one(backbone, seed, data_root, batch_size=32, epochs=50):
    set_seed(seed)
    output_dir = ROOT / "experiments" / "taskB" / backbone / f"seed_{seed}"
    output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, val_loader, _ = get_dataloaders(
        data_root=data_root, batch_size=batch_size, num_workers=4, task="taskB"
    )
    num_classes = len(train_loader.dataset.classes)
    print(f"  [{backbone} s{seed}] train={len(train_loader.dataset)}, classes={num_classes}, batch={batch_size}")

    run = wandb.init(
        project="woodbench-taskB",
        name=f"{backbone}_s{seed}_machine-A",
        config={
            "backbone": backbone, "seed": seed, "epochs": epochs,
            "batch_size": batch_size, "num_classes": num_classes,
            "lr_head": 1e-4, "lr_backbone": 1e-5,
            "label_smoothing": 0.1, "scheduler": "cosine", "amp": True,
            "machine": "A", "parallel": False,
        },
        dir=str(output_dir),
    )

    model = create_model(backbone, num_classes=num_classes, pretrained=True, freeze_backbone=False)
    optimizer = create_optimizer(model, lr=1e-4, weight_decay=1e-4, backbone_lr=1e-5)
    scheduler = create_scheduler(optimizer, epochs=epochs, scheduler_type="cosine")
    criterion = create_loss_function("label_smoothing", label_smoothing=0.1)

    engine = TrainingEngine(
        model=model, train_loader=train_loader, val_loader=val_loader,
        optimizer=optimizer, scheduler=scheduler, criterion=criterion,
        device=DEVICE, use_amp=True,
    )
    result = engine.fit(epochs=epochs, output_dir=output_dir,
                       name=f"{backbone}_seed{seed}", save_best_only=False)
    run.finish()
    print(f"  [{backbone} s{seed}] val_acc={result['best_val_acc']:.2f}%")
    return result

In [ ]:
EFFICIENTNET_SEEDS = [42, 123, 456]
eff_results = {}

for seed in EFFICIENTNET_SEEDS:
    print(f"\n{'='*60}")
    print(f"EfficientNet-B3 | seed={seed} ({EFFICIENTNET_SEEDS.index(seed)+1}/{len(EFFICIENTNET_SEEDS)})")
    print(f"{'='*60}")
    result = train_one("efficientnet_b3", seed, str(DATA_ROOT), batch_size=32, epochs=50)
    eff_results[seed] = result

print("\n" + "="*60)
print("EfficientNet-B3 Summary:")
for seed, res in eff_results.items():
    print(f"  seed={seed}: {res['best_val_acc']:.2f}%")
accs = [r['best_val_acc'] for r in eff_results.values()]
print(f"  Mean: {np.mean(accs):.2f}% ± {np.std(accs):.2f}%")